# Import Libraries

In [19]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import CSVLoader,PyPDFLoader
from langchain_community.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.messages import SystemMessage
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from huggingface_hub import login
from sentence_transformers import CrossEncoder
from langchain_community.llms import OpenAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv
import os
import time
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import re
from langchain_core.documents import Document
from torch import embedding

# Init the model 

In [58]:
load_dotenv('secret.env')
key = os.getenv("Open_Ai_Key")
model = ChatOpenAI(
    model_name="openai/gpt-oss-120b:free",
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=key,
)

# Load Data 

In [10]:
DATA_DIR = Path("c:/Users/tabarak/VSCODE/University/NLP Project/Data")
DATA_DIR.mkdir(exist_ok=True) 

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/120.0.0.0 Safari/537.36"
}


URLS = {

    # ── ASTHMA ──────────────────────────────────────────────
    "asthma_who" : "https://www.who.int/news-room/fact-sheets/detail/asthma",
    "asthma_nih" : "https://medlineplus.gov/asthma.html",
    "asthma_nhs" : "https://www.nhs.uk/conditions/asthma/",

    # ── CANCER ──────────────────────────────────────────────
    "cancer_who" : "https://www.who.int/news-room/fact-sheets/detail/cancer",
    "cancer_nih" : "https://medlineplus.gov/cancer.html",
    "cancer_nhs" : "https://www.nhs.uk/conditions/cancer/",

    # ── COVID-19 ────────────────────────────────────────────
    "covid19_who" : "https://www.who.int/news-room/fact-sheets/detail/coronavirus-disease-(covid-19)",
    "covid19_nhs" : "https://www.nhs.uk/conditions/covid-19/",

    # ── DENGUE ──────────────────────────────────────────────
    "dengue_who" : "https://www.who.int/news-room/fact-sheets/detail/dengue-and-severe-dengue",
    "dengue_nih" : "https://medlineplus.gov/dengue.html",
    "dengue_nhs" : "https://www.nhs.uk/conditions/dengue/",

    # ── DIABETES ────────────────────────────────────────────
    "diabetes_who" : "https://www.who.int/news-room/fact-sheets/detail/diabetes",
    "diabetes_nih" : "https://medlineplus.gov/diabetes.html",
    "diabetes_nhs" : "https://www.nhs.uk/conditions/diabetes/",

    # ── HEART DISEASE ───────────────────────────────────────
    "heart_disease_who" : "https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)",
    "heart_disease_nih" : "https://medlineplus.gov/heartdiseases.html",
    "heart_disease_nhs" : "https://www.nhs.uk/conditions/coronary-heart-disease/",
    "heart_disease_cdc" : "https://www.cdc.gov/heartdisease/index.htm",

    # ── HYPERTENSION ────────────────────────────────────────
    "hypertension_who" : "https://www.who.int/news-room/fact-sheets/detail/hypertension",
    "hypertension_nih" : "https://medlineplus.gov/highbloodpressure.html",
    "hypertension_nhs" : "https://www.nhs.uk/conditions/high-blood-pressure-hypertension/",

    # ── MALARIA ─────────────────────────────────────────────
    "malaria_who" : "https://www.who.int/news-room/fact-sheets/detail/malaria",
    "malaria_nih" : "https://medlineplus.gov/malaria.html",
    "malaria_nhs" : "https://www.nhs.uk/conditions/malaria/",

    # ── STROKE ──────────────────────────────────────────────
    "stroke_who" : "https://www.who.int/news-room/fact-sheets/detail/the-top-10-causes-of-death",
    "stroke_nih" : "https://medlineplus.gov/stroke.html",
    "stroke_nhs" : "https://www.nhs.uk/conditions/stroke/",

    # ── TUBERCULOSIS ────────────────────────────────────────
    "tuberculosis_who" : "https://www.who.int/news-room/fact-sheets/detail/tuberculosis",
    "tuberculosis_nih" : "https://medlineplus.gov/tuberculosis.html",
    "tuberculosis_nhs" : "https://www.nhs.uk/conditions/tuberculosis-tb/",
    "tuberculosis_cdc" : "https://www.cdc.gov/tb/index.html",
}
# ============================================================
# SCRAPER FUNCTION
# ============================================================

def scrape_page(url: str) -> str | None:
    """
    Scrape a URL and return clean text content.
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # ── Remove junk tags ──────────────────────────────────
        for tag in soup(["script", "style", "nav", "footer",
                          "header", "aside", "form", "button",
                          "iframe", "noscript"]):
            tag.decompose()

        # ── Extract main content ──────────────────────────────
        # Try to find the main article/content area
        main = (
            soup.find("main")
            or soup.find("article")
            or soup.find("div", {"id"   : "main-content"})
            or soup.find("div", {"class": "content"})
            or soup.find("div", {"class": "main"})
            or soup.body
        )

        if main:
            # Get all text, clean whitespace
            lines = []
            for element in main.find_all(
                ["h1","h2","h3","h4","p","li","td","th"]
            ):
                text = element.get_text(separator=" ", strip=True)
                if text and len(text) > 30:   # skip tiny fragments
                    lines.append(text)

            return "\n\n".join(lines)

        return None

    except requests.exceptions.Timeout:
        print(f"    ⏱️  Timeout")
        return None
    except requests.exceptions.HTTPError as e:
        print(f"    🚫 HTTP Error: {e}")
        return None
    except Exception as e:
        print(f"    ❌ Error: {e}")
        return None


# ============================================================
# SAVE FUNCTION
# ============================================================

def save_text(filename: str, content: str) -> None:
    """
    Save text content to /data folder.
    """
    filepath = DATA_DIR / f"{filename}.txt"
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)


# ============================================================
# MAIN COLLECTOR
# ============================================================

def collect_all() -> None:

    success = []
    failed  = []

    print("=" * 60)
    print("🌐  COLLECTING ALL DISEASE DATA")
    print(f"📂  Saving to : {DATA_DIR}")
    print(f"📄  Total URLs: {len(URLS)}")
    print("=" * 60)

    for name, url in URLS.items():

        disease, source = name.rsplit("_", 1)
        print(f"\n🦠  {disease.upper()} | 📚 {source.upper()}")
        print(f"    🔗 {url}")

        content = scrape_page(url)

        if content and len(content) > 100:
            save_text(name, content)
            success.append(name)
            print(f"    ✅ Saved → {name}.txt ({len(content):,} chars)")
        else:
            failed.append(name)
            print(f"    ❌ Failed or empty → skipping")

        # ── Be polite to servers ──────────────────────────────
        time.sleep(2)

    # ── Final Summary ─────────────────────────────────────────
    print("\n" + "=" * 60)
    print("📊  COLLECTION SUMMARY")
    print("=" * 60)
    print(f"  ✅ Success : {len(success)} files saved")
    print(f"  ❌ Failed  : {len(failed)} files")

    if failed:
        print(f"\n  ⚠️  Failed URLs:")
        for f in failed:
            print(f"     ✗ {f}")

    print(f"\n  📂 All files saved in: {DATA_DIR}")
    print("=" * 60)

In [11]:
collect_all()

🌐  COLLECTING ALL DISEASE DATA
📂  Saving to : c:\Users\tabarak\VSCODE\University\NLP Project\Data
📄  Total URLs: 31

🦠  ASTHMA | 📚 WHO
    🔗 https://www.who.int/news-room/fact-sheets/detail/asthma
    ✅ Saved → asthma_who.txt (7,550 chars)

🦠  ASTHMA | 📚 NIH
    🔗 https://medlineplus.gov/asthma.html
    ✅ Saved → asthma_nih.txt (11,927 chars)

🦠  ASTHMA | 📚 NHS
    🔗 https://www.nhs.uk/conditions/asthma/
    ✅ Saved → asthma_nhs.txt (9,542 chars)

🦠  CANCER | 📚 WHO
    🔗 https://www.who.int/news-room/fact-sheets/detail/cancer
    ✅ Saved → cancer_who.txt (12,266 chars)

🦠  CANCER | 📚 NIH
    🔗 https://medlineplus.gov/cancer.html
    ✅ Saved → cancer_nih.txt (11,579 chars)

🦠  CANCER | 📚 NHS
    🔗 https://www.nhs.uk/conditions/cancer/
    ✅ Saved → cancer_nhs.txt (8,120 chars)

🦠  COVID19 | 📚 WHO
    🔗 https://www.who.int/news-room/fact-sheets/detail/coronavirus-disease-(covid-19)
    ✅ Saved → covid19_who.txt (21,751 chars)

🦠  COVID19 | 📚 NHS
    🔗 https://www.nhs.uk/conditions/covid-

## Load Data into one structured file 

In [12]:
AVAILABLE_FILES = [

    # ── ASTHMA ──────────────────────────────────────────────
    "asthma_nhs",
    "asthma_nih",
    "asthma_who",

    # ── CANCER ──────────────────────────────────────────────
    "cancer_nhs",
    "cancer_nih",
    "cancer_who",

    # ── COVID-19 ────────────────────────────────────────────
    "covid19_nhs",
    "covid19_who",

    # ── DENGUE ──────────────────────────────────────────────
    "dengue_nhs",
    "dengue_nih",
    "dengue_who",

    # ── DIABETES ────────────────────────────────────────────
    "diabetes_nhs",
    "diabetes_nih",
    "diabetes_who",

    # ── HEART DISEASE ───────────────────────────────────────
    "heart_disease_cdc",
    "heart_disease_nhs",
    "heart_disease_nih",
    "heart_disease_who",

    # ── HYPERTENSION ────────────────────────────────────────
    "hypertension_nhs",
    "hypertension_nih",
    "hypertension_who",

    # ── MALARIA ─────────────────────────────────────────────
    "malaria_nhs",
    "malaria_nih",
    "malaria_who",

    # ── STROKE ──────────────────────────────────────────────
    "stroke_nhs",
    "stroke_nih",
    "stroke_who",

    # ── TUBERCULOSIS ────────────────────────────────────────
    "tuberculosis_cdc",
    "tuberculosis_nhs",
    "tuberculosis_nih",
    "tuberculosis_who",
]

# ============================================================
# LOAD SINGLE FILE
# ============================================================

def load_single_file(filepath: Path) -> dict | None:
    """
    Load a single .txt file and return a structured dict.
    """
    if not filepath.exists():
        print(f"  ⚠️  Missing : {filepath.name}")
        return None

    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    if not content.strip():
        print(f"  ⚠️  Empty   : {filepath.name}")
        return None

    # ── Parse disease + source from filename ─────────────────
    stem    = filepath.stem          # e.g. "heart_disease_who"
    parts   = stem.split("_")
    source  = parts[-1]              # "who"
    disease = "_".join(parts[:-1])   # "heart_disease"

    return {
        "disease"  : disease,
        "source"   : source,
        "filename" : filepath.name,
        "filepath" : str(filepath),
        "content"  : content.strip(),
        "length"   : len(content.strip())
    }

# ============================================================
# LOAD ALL DOCUMENTS
# ============================================================

def load_all_documents() -> list[dict]:
    """
    Load all 31 available .txt files from /data directory.
    Returns a list of document dicts.
    """
    documents       = []
    current_disease = None

    print("=" * 60)
    print("📥  LOADING ALL DISEASE DOCUMENTS")
    print(f"📂  Data folder : {DATA_DIR}")
    print(f"📄  Total files : {len(AVAILABLE_FILES)}")
    print("=" * 60)

    for name in AVAILABLE_FILES:
        filepath = DATA_DIR / f"{name}.txt"

        # ── Parse for display ─────────────────────────────────
        parts   = name.split("_")
        source  = parts[-1]
        disease = "_".join(parts[:-1])

        # ── Print disease header when it changes ──────────────
        if disease != current_disease:
            current_disease = disease
            print(f"\n🦠  {disease.upper()}")

        doc = load_single_file(filepath)

        if doc:
            documents.append(doc)
            print(f"  ✅ {source.upper():<6} → {doc['length']:>7,} chars")

    # ── Summary ──────────────────────────────────────────────
    total_chars = sum(d["length"] for d in documents)

    print("\n" + "=" * 60)
    print("📊  LOADING SUMMARY")
    print("=" * 60)
    print(f"  📄 Total documents  : {len(documents)}")
    print(f"  📝 Total characters : {total_chars:,}")
    print(f"  📝 Avg chars / doc  : {total_chars // max(len(documents), 1):,}")
    print("=" * 60)

    return documents

In [13]:
all_docs = load_all_documents()

📥  LOADING ALL DISEASE DOCUMENTS
📂  Data folder : c:\Users\tabarak\VSCODE\University\NLP Project\Data
📄  Total files : 31

🦠  ASTHMA
  ✅ NHS    →   9,542 chars
  ✅ NIH    →  11,927 chars
  ✅ WHO    →   7,550 chars

🦠  CANCER
  ✅ NHS    →   8,120 chars
  ✅ NIH    →  11,579 chars
  ✅ WHO    →  12,266 chars

🦠  COVID19
  ✅ NHS    →     458 chars
  ✅ WHO    →  21,751 chars

🦠  DENGUE
  ✅ NHS    →   4,197 chars
  ✅ NIH    →   4,321 chars
  ✅ WHO    →  11,132 chars

🦠  DIABETES
  ✅ NHS    →   3,684 chars
  ✅ NIH    →  11,364 chars
  ✅ WHO    →   7,293 chars

🦠  HEART_DISEASE
  ✅ CDC    →   2,331 chars
  ✅ NHS    →   3,359 chars
  ✅ NIH    →  10,555 chars
  ✅ WHO    →   9,843 chars

🦠  HYPERTENSION
  ✅ NHS    →   5,754 chars
  ✅ NIH    →   8,815 chars
  ✅ WHO    →   8,708 chars

🦠  MALARIA
  ✅ NHS    →   3,847 chars
  ✅ NIH    →   2,533 chars
  ✅ WHO    →  24,875 chars

🦠  STROKE
  ✅ NHS    →     224 chars
  ✅ NIH    →  11,411 chars
  ✅ WHO    →  10,064 chars

🦠  TUBERCULOSIS
  ✅ CDC    →   1

## preprocess data 

In [15]:
def fix_encoding(text: str) -> str:
    """
    Fix common encoding artifacts found in your files.
    Found in 22/31 files.
    """
    encoding_map = {
        "\u00a0": " ",    # non-breaking space
        "\u2019": "'",    # curly apostrophe    →  '
        "\u2018": "'",    # curly quote         →  '
        "\u201c": '"',    # left double quote   →  "
        "\u201d": '"',    # right double quote  →  "
        "\u2013": "-",    # en dash             →  -
        "\u2014": "-",    # em dash             →  -
    }
    for bad, good in encoding_map.items():
        text = text.replace(bad, good)
    return text

# ============================================================
# STEP 2 — REMOVE URLs
# ============================================================

def remove_urls(text: str) -> str:
    """
    Remove any line that contains a URL — whole line removed.
    Found in 5/31 files. Only 2 URLs in your actual data.
    """
    lines = text.split("\n")
    lines = [
        line for line in lines
        if not re.search(r"http\S+|www\.\S+", line)
    ]
    return "\n".join(lines)

# ============================================================
# STEP 3 — REMOVE SPECIAL CHARACTERS
# ============================================================

def remove_special_chars(text: str) -> str:
    """
    Remove special symbols found in your files.
    Found in 2/31 files (covid19_who, tuberculosis_who).
    Keeps normal punctuation → . , ! ? : ; ( ) - ' "
    """
    text = re.sub(r"[|~^•*#@<>{}\\]", " ", text)
    return text

# ============================================================
# STEP 4 — REMOVE SHORT LINES
# ============================================================

def remove_short_lines(text: str) -> str:
    """
    Remove lines that are too short to be meaningful.
    Found in 25/31 files (navigation/menu leftovers).
    Threshold = 40 characters.
    """
    lines = text.split("\n")
    lines = [
        line for line in lines
        if len(line.strip()) == 0    # keep blank lines (paragraph breaks)
        or len(line.strip()) >= 40   # keep long meaningful lines
    ]
    return "\n".join(lines)

# ============================================================
# STEP 5 — FIX EXTRA SPACES & NORMALIZE WHITESPACE
# ============================================================

def fix_whitespace(text: str) -> str:
    """
    Fix extra spaces and normalize blank lines.
    Found in 9/31 files.
    """
    # Collapse multiple spaces → single space
    text = re.sub(r" {2,}", " ", text)

    # Normalize line endings
    text = re.sub(r"\r\n|\r", "\n", text)

    # Collapse more than 2 blank lines → single blank line
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Strip each line
    lines = [line.strip() for line in text.split("\n")]
    text  = "\n".join(lines)

    # Final strip
    return text.strip()

# ============================================================
# MAIN PREPROCESS FUNCTION
# ============================================================

def preprocess(text: str) -> str:
    """
    Full cleaning pipeline:
    1. Fix encoding
    2. Remove URLs
    3. Remove special chars
    4. Remove short lines
    5. Fix whitespace
    """
    text = fix_encoding(text)
    text = remove_urls(text)
    text = remove_special_chars(text)
    text = remove_short_lines(text)
    text = fix_whitespace(text)
    return text

# ============================================================
# PREPROCESS ALL DOCUMENTS
# ============================================================

def preprocess_all(documents: list[dict]) -> list[dict]:
    """
    Apply cleaning to all loaded documents.
    """
    print("=" * 60)
    print("🧹  PREPROCESSING ALL DOCUMENTS")
    print("=" * 60)

    current_disease = None

    for doc in documents:
        if doc["disease"] != current_disease:
            current_disease = doc["disease"]
            print(f"\n🦠  {doc['disease'].upper()}")

        raw_length   = len(doc["content"])
        clean_text   = preprocess(doc["content"])
        clean_length = len(clean_text)

        doc["content"] = clean_text
        doc["length"]  = clean_length

        print(f"  ✅ {doc['source'].upper():<6}"
              f" | before: {raw_length:>7,}"
              f" → after: {clean_length:>7,}"
              f" | removed: {raw_length - clean_length:>5,} chars")

    total_clean = sum(d["length"] for d in documents)
    print("\n" + "=" * 60)
    print("📊  CLEANING SUMMARY")
    print("=" * 60)
    print(f"  📄 Total documents  : {len(documents)}")
    print(f"  ✨ Total clean chars: {total_clean:,}")
    print("=" * 60)

    return documents

In [16]:
clean_docs = preprocess_all(all_docs)

🧹  PREPROCESSING ALL DOCUMENTS

🦠  ASTHMA
  ✅ NHS    | before:   9,542 → after:   9,134 | removed:   408 chars
  ✅ NIH    | before:  11,927 → after:  11,708 | removed:   219 chars
  ✅ WHO    | before:   7,550 → after:   7,433 | removed:   117 chars

🦠  CANCER
  ✅ NHS    | before:   8,120 → after:   7,675 | removed:   445 chars
  ✅ NIH    | before:  11,579 → after:  11,399 | removed:   180 chars
  ✅ WHO    | before:  12,266 → after:  11,605 | removed:   661 chars

🦠  COVID19
  ✅ NHS    | before:     458 → after:     157 | removed:   301 chars
  ✅ WHO    | before:  21,751 → after:  10,118 | removed: 11,633 chars

🦠  DENGUE
  ✅ NHS    | before:   4,197 → after:   3,935 | removed:   262 chars
  ✅ NIH    | before:   4,321 → after:   4,169 | removed:   152 chars
  ✅ WHO    | before:  11,132 → after:  10,924 | removed:   208 chars

🦠  DIABETES
  ✅ NHS    | before:   3,684 → after:   3,647 | removed:    37 chars
  ✅ NIH    | before:  11,364 → after:  10,921 | removed:   443 chars
  ✅ WHO    | 

# Data Chunking 

In [17]:
MIN_CHUNK      = 150   
MAX_CHUNK      = 800   
TARGET_SENTENCES = 4   

def split_into_chunks(text: str) -> list[str]:
    """
    Split text into semantic chunks using sentence grouping.
    Groups sentences into chunks of ~4 sentences each.
    Perfect for short-paragraph medical text.
    """

    # ── Handle tiny files → keep as one chunk ────────────
    if len(text) < MIN_CHUNK:
        return [text]

    # ── Step 1: Split into sentences ─────────────────────
    # First normalize newlines to spaces within paragraphs
    paragraphs = [p.strip() for p in re.split(r"\n\n+", text)]
    paragraphs = [p for p in paragraphs if len(p) > 0]

    # Join paragraphs with space, split into sentences
    all_sentences = []
    for para in paragraphs:
        sentences = re.split(r"(?<=[.!?])\s+", para.strip())
        sentences = [s.strip() for s in sentences if len(s.strip()) > 20]
        all_sentences.extend(sentences)

    if not all_sentences:
        return [text]

    # ── Step 2: Group sentences into chunks ───────────────
    chunks  = []
    current = []
    current_len = 0

    for sentence in all_sentences:
        current.append(sentence)
        current_len += len(sentence)

        # Save chunk when we hit target size or max chars
        if len(current) >= TARGET_SENTENCES or current_len >= MAX_CHUNK:
            chunk_text = " ".join(current).strip()
            if len(chunk_text) >= MIN_CHUNK:
                chunks.append(chunk_text)
            elif chunks:
                # Too small → merge with previous chunk
                chunks[-1] = chunks[-1] + " " + chunk_text
            else:
                chunks.append(chunk_text)
            current     = []
            current_len = 0

    # ── Save remaining sentences ──────────────────────────
    if current:
        chunk_text = " ".join(current).strip()
        if chunks and len(chunk_text) < MIN_CHUNK:
            # Too small → merge with previous
            chunks[-1] = chunks[-1] + " " + chunk_text
        else:
            chunks.append(chunk_text)

    # ── Final safety check ────────────────────────────────
    if not chunks:
        return [text]

    return chunks

def chunk_document(doc: dict) -> list[dict]:
    """
    Chunk a single document and attach metadata to each chunk.
    """
    chunks     = split_into_chunks(doc["content"])
    chunk_list = []

    for i, chunk_text in enumerate(chunks):
        chunk_list.append({
            "chunk_id" : f"{doc['disease']}_{doc['source']}_{i}",
            "disease"  : doc["disease"],
            "source"   : doc["source"],
            "content"  : chunk_text,
            "length"   : len(chunk_text)
        })

    return chunk_list


def chunk_all(documents: list[dict]) -> list[dict]:
    """
    Chunk ALL documents and return flat list of all chunks.
    """
    all_chunks      = []
    current_disease = None

    print("=" * 60)
    print("✂️   SEMANTIC CHUNKING ALL DOCUMENTS")
    print(f"     MIN chunk size : {MIN_CHUNK} chars")
    print(f"     MAX chunk size : {MAX_CHUNK} chars")
    print("=" * 60)

    for doc in documents:
        if doc["disease"] != current_disease:
            current_disease = doc["disease"]
            print(f"\n🦠  {doc['disease'].upper()}")

        chunks = chunk_document(doc)
        all_chunks.extend(chunks)

        avg_size = sum(c["length"] for c in chunks) // max(len(chunks), 1)

        print(f"  ✅ {doc['source'].upper():<6}"
              f" | chunks: {len(chunks):>3}"
              f" | avg size: {avg_size:>5} chars")

    # ── Summary ──────────────────────────────────────────────
    total_chunks = len(all_chunks)
    avg_chunk    = sum(c["length"] for c in all_chunks) // max(total_chunks, 1)

    print("\n" + "=" * 60)
    print("📊  CHUNKING SUMMARY")
    print("=" * 60)
    print(f"  ✂️  Total chunks     : {total_chunks}")
    print(f"  📏 Avg chunk size   : {avg_chunk} chars")
    print(f"  📄 Total documents  : {len(documents)}")
    print("=" * 60)

    return all_chunks

In [18]:
chunks = chunk_all(clean_docs)

✂️   SEMANTIC CHUNKING ALL DOCUMENTS
     MIN chunk size : 150 chars
     MAX chunk size : 800 chars

🦠  ASTHMA
  ✅ NHS    | chunks:  25 | avg size:   360 chars
  ✅ NIH    | chunks:  39 | avg size:   293 chars
  ✅ WHO    | chunks:  18 | avg size:   409 chars

🦠  CANCER
  ✅ NHS    | chunks:  23 | avg size:   329 chars
  ✅ NIH    | chunks:  38 | avg size:   294 chars
  ✅ WHO    | chunks:  26 | avg size:   441 chars

🦠  COVID19
  ✅ NHS    | chunks:   1 | avg size:   157 chars
  ✅ WHO    | chunks:  20 | avg size:   501 chars

🦠  DENGUE
  ✅ NHS    | chunks:  12 | avg size:   321 chars
  ✅ NIH    | chunks:  16 | avg size:   257 chars
  ✅ WHO    | chunks:  24 | avg size:   450 chars

🦠  DIABETES
  ✅ NHS    | chunks:  11 | avg size:   327 chars
  ✅ NIH    | chunks:  33 | avg size:   324 chars
  ✅ WHO    | chunks:  19 | avg size:   367 chars

🦠  HEART_DISEASE
  ✅ CDC    | chunks:   7 | avg size:   302 chars
  ✅ NHS    | chunks:   9 | avg size:   343 chars
  ✅ NIH    | chunks:  34 | avg size:   

# Embeddings Creation

## Build Embedding Model For both query and paragraph

In [69]:
def build_emb_model(embedding_type: str = "paragraph"):
    model = "sentence-transformers/all-MiniLM-L6-v2"
    if embedding_type=="paragraph":
        encode_kwargs = {"normalize_embeddings": True}
        model_kwargs = {"device": "cpu"}
        embedding_model = HuggingFaceEmbeddings(
            model_name=model,
            model_kwargs=model_kwargs,
            encode_kwargs=encode_kwargs,
        )
    elif embedding_type=="query":
        encode_kwargs = {"normalize_embeddings": True}
        model_kwargs = {"device": "cpu"}
        embedding_model = HuggingFaceEmbeddings(
            model_name=model,
            model_kwargs=model_kwargs,
            encode_kwargs=encode_kwargs,
        )
    return embedding_model

## Build Embeddings for all docs 

In [70]:
def build_vector_db():
    docs = [
        Document(
            page_content=chunk["content"],
            metadata={
                "disease": chunk["disease"],
                "source": chunk["source"],
                "chunk_id": chunk["chunk_id"]
            }
        )
        for chunk in chunks
    ]
    embedding_model = build_emb_model(embedding_type="paragraph")
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embedding_model,
        persist_directory="./chroma_db"
    )
    return vectorstore

In [26]:
vector_store = build_vector_db()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 699.08it/s]


# Build and embed user query 

## query rewrite

In [71]:
def query_rewrite(query):
  template = PromptTemplate(
      input_variables=["query"],
      template=(
          "You are an expert in information retrieval.\n"
          "Rewrite the following user query to be more specific, detailed, "
          "and aligned with terminology likely used in technical and regulatory documents.\n"
          "Do NOT answer the query.\n"
          "Return ONLY the rewritten query.\n\n"
          "Original query: {query}"
          )
  )
  chain = template|model
  return chain.invoke({"query":query})

## Embed Query

In [72]:
def build_user_query(query:str):
    embedding_model = build_emb_model(embedding_type="query")
    query_vector = embedding_model.embed_query(query)
    return query_vector

# Retrive Docs 

## retrive docs function 

In [80]:
def retrive_docs(vector_store, query):
    Retrival = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3},
    )
    docs = Retrival.invoke(query)
    return docs

## Rerank docs based on cross encoder 

In [81]:
def Rerank_docs(query, vector_store):
    docs = retrive_docs(vector_store, query)
    Reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L12-v2")
    pairs = [(query, d.page_content) for d in docs]
    scores = Reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked]

# Build Response 

In [82]:
template = ChatPromptTemplate.from_messages([
    ("system", "You are a precise assistant. Answer ONLY using the provided context."),
    ("human",
     "Context:\n{context}\n\n"
     "Question:\n{question}")
])


In [83]:
def Build_response(query, vector_store):
    top_docs = Rerank_docs(query, vector_store)
    context = "\n\n".join(d.page_content for d in top_docs)
    rag_chain = template | model
    response = rag_chain.invoke({
        "context": context,
        "question": query,
    })
    return response.content

# Test pipeline

In [87]:
query = "What is the symptoms of diabities?"

In [88]:
query = query_rewrite(query).content
query_emb = build_user_query(query)
response = Build_response(query, vector_store)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1604.32it/s]


In [89]:
print(f"model response: {response}")

model response: **Clinical manifestations**

* **Polyuria** – needing to urinate more often than usual is a frequent early sign.  
* **Gradual or mild symptoms** – in type 2 diabetes the classic signs (e.g., increased thirst, weight loss, fatigue) are often subtle and may go unnoticed for years.  
* **Potential complications** – because the disease can be present for a long time before diagnosis, damage to blood vessels may already be occurring in the heart, eyes, kidneys and peripheral nerves.  

**Diagnostic considerations**

* The diagnosis is usually made after the patient reports the above symptoms and/or after routine screening because the manifestations can be very mild.  
* Laboratory confirmation (e.g., fasting plasma glucose, oral‑glucose‑tolerance test, HbA1c) is required, but the specific numeric criteria are not provided in the given material.  

Thus, the typical clinical picture of diabetes mellitus includes polyuria, often‑subtle systemic symptoms, and evidence of vascu

## 